In [ ]:
"""
exp298: H-085 TabPFN Variant D(Variant C+H-075結合TE12列)の軽量スクリーニング — Kaggle Notebook自己完結版。

exp294/295(Variant C, raw13+te_exact39=52列, val_ba=0.95061/OOF=0.95031)を確立済み。
H-075(結合TargetEncoding: sleep×stress×activityの3-way+2-wayペア3組=12列)はLGB/FT-Transformer
で棄却済みだが、棄却理由がアーキ固有(木のsplit自力学習/embedding競合)であり、TabPFNの
in-context learning機構には当てはまらない可能性がある(指針#13、早期却下の禁止)。

sleep_durationはルール閾値(<6,6-7,>=7)でbin化してから結合。結合TEはH-070と同一手法
(TargetEncoder(cv=5,smooth="auto"))でfold-safeに生成。

コスト制約のため単一80/20層化分割でOOF方向性を確認する。
出力: /kaggle/working/tabpfn_variantD_screen_result.json
"""
import subprocess, sys, time, json
from pathlib import Path

t0 = time.time()


def _gpu_name():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=10)
        return out.stdout.strip()
    except Exception:
        return ""


print("detected GPU:", _gpu_name())

import os
_candidate_cache_dirs = [
    "/kaggle/input/models/prior-labsai/tabpfn-3/PyTorch/default/1",
    "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1",
]
for _d in _candidate_cache_dirs:
    if Path(_d).exists():
        os.environ["TABPFN_MODEL_CACHE_DIR"] = _d
        print("TABPFN_MODEL_CACHE_DIR =", _d)
        break
else:
    print("警告: マウント済みモデルディレクトリが見つからない。候補:", _candidate_cache_dirs)
    subprocess.run(["find", "/kaggle/input", "-maxdepth", "6"], check=False)

try:
    import tabpfn  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)

if "P100" in _gpu_name():
    import torch as _torch_check
    _torch_ver = _torch_check.__version__.split("+")[0]
    print(f"installed torch version: {_torch_ver} -> reinstalling cu118 build for P100/Pascal support")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     f"torch=={_torch_ver}", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import TargetEncoder
from tabpfn import TabPFNClassifier

TARGET_COL = "health_condition"
SEED = 42

_KAGGLE_INPUT = Path("/kaggle/input")
_candidates = [
    _KAGGLE_INPUT / "competitions" / "playground-series-s6e7",
    _KAGGLE_INPUT / "playground-series-s6e7",
]
DATA_DIR = next((p for p in _candidates if (p / "train.csv").exists()), _candidates[0])
print("DATA_DIR =", DATA_DIR)

train = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
test = pd.read_csv(DATA_DIR / "test.csv", index_col="id")
print(f"train: {train.shape}, test: {test.shape}")

y_raw = train[TARGET_COL]
X_raw13 = train.drop(columns=[TARGET_COL])
X_test_raw13 = test.copy()
classes = np.array(sorted(y_raw.unique()))
RAW13_COLS = list(X_raw13.columns)

# ---- Variant C: H-070単変数TE(39列) ----
print("H-070厳密値TE(39列)を生成中...")
te_train_cols, te_test_cols = {}, {}
for col in RAW13_COLS:
    enc = TargetEncoder(cv=5, shuffle=True, random_state=SEED,
                         smooth="auto", target_type="multiclass")
    Z_tr = enc.fit_transform(X_raw13[[col]], y_raw)
    Z_te = enc.transform(X_test_raw13[[col]])
    for k, c in enumerate(enc.classes_):
        cname = f"te_exact_{col}_{c}"
        te_train_cols[cname] = Z_tr[:, k]
        te_test_cols[cname] = Z_te[:, k]
print(f"H-070 TE列生成完了: {len(te_train_cols)}列")

# ---- H-075: 結合TE(3-way + 2-wayペア3組 = 12列) ----
print("H-075結合TE(12列)を生成中...")
alldf = pd.concat([X_raw13, X_test_raw13], ignore_index=True)
n_tr = len(X_raw13)
sd = alldf["sleep_duration"]
bucket_vals = np.select([sd < 6, sd < 7, sd >= 7], ["lt6", "6to7", "ge7"], default="__nan__")
sleep_bucket = pd.Series(bucket_vals, index=alldf.index)
sleep_bucket[sd.isna()] = np.nan
stress = alldf["stress_level"]
activity = alldf["physical_activity_level"]


def s(col):
    return col.astype(str).where(col.notna(), "__nan__")


sleep_s, stress_s, activity_s = s(sleep_bucket), s(stress), s(activity)
triple = sleep_s + "|" + stress_s + "|" + activity_s
pair_ss = sleep_s + "|" + stress_s
pair_sa = sleep_s + "|" + activity_s
pair_st = stress_s + "|" + activity_s

joint_train_cols, joint_test_cols = {}, {}
for name, series in [("triple_sleep_stress_activity", triple),
                      ("pair_sleep_stress", pair_ss),
                      ("pair_sleep_activity", pair_sa),
                      ("pair_stress_activity", pair_st)]:
    enc = TargetEncoder(cv=5, shuffle=True, random_state=SEED,
                         smooth="auto", target_type="multiclass")
    Z_tr = enc.fit_transform(series.iloc[:n_tr].to_frame(name), y_raw)
    Z_te = enc.transform(series.iloc[n_tr:].to_frame(name))
    for k, c in enumerate(enc.classes_):
        cname = f"te_{name}_{c}"
        joint_train_cols[cname] = Z_tr[:, k]
        joint_test_cols[cname] = Z_te[:, k]
print(f"H-075 結合TE列生成完了: {len(joint_train_cols)}列")

X = X_raw13.copy()
for cname, vals in {**te_train_cols, **joint_train_cols}.items():
    X[cname] = vals
print(f"Variant D 最終特徴量数: {X.shape[1]} (raw13 + te_exact39 + joint_te12)")

prior = y_raw.value_counts(normalize=True).reindex(classes).to_numpy()

X_tr, X_val, y_tr, y_val = train_test_split(X, y_raw, test_size=0.2, stratify=y_raw, random_state=SEED)
print(f"X_tr: {X_tr.shape}, X_val: {X_val.shape}")

n_gpus = torch.cuda.device_count()
print("torch.cuda.device_count() =", n_gpus)
device_arg = [f"cuda:{i}" for i in range(n_gpus)] if n_gpus > 1 else "cuda"
print("device_arg =", device_arg)

clf = TabPFNClassifier(
    device=device_arg,
    n_estimators=2,
    balance_probabilities=True,
    fit_mode="fit_with_cache",
)
clf.fit(X_tr, y_tr)
print(f"fit完了 ({(time.time()-t0)/60:.1f}min)")

val_proba = clf.predict_proba(X_val)
tabpfn_classes = clf.classes_
c2i_tp = {c: i for i, c in enumerate(tabpfn_classes)}
y_val_codes = y_val.map(c2i_tp).to_numpy()
raw_acc = balanced_accuracy_score(y_val_codes, val_proba.argmax(1))
print(f"raw argmax val balanced_accuracy = {raw_acc:.5f}")

prior_ordered = pd.Series(y_raw).value_counts(normalize=True).reindex(tabpfn_classes).to_numpy()
results = {"raw": float(raw_acc)}
best = (raw_acc, 0.0)
for b in [0.0, 0.25, 0.5, 0.75, 1.0]:
    p = val_proba / prior_ordered ** b
    p = p / p.sum(1, keepdims=True)
    s = balanced_accuracy_score(y_val_codes, p.argmax(1))
    results[f"beta_{b}"] = float(s)
    print(f"beta={b:>4}: calibrated val balanced_accuracy = {s:.5f}")
    if s > best[0]:
        best = (s, b)

print(f"\nbest beta={best[1]}, best val balanced_accuracy={best[0]:.5f}")
print("(参考: Variant C(52列) 単一split val_ba=0.95061/0.95076, フル5-fold OOF=0.95031)")

output = {
    "variant": "D_raw13_te39_jointte12",
    "n_features": int(X.shape[1]),
    "val_balanced_accuracy_raw": float(raw_acc),
    "best_beta": best[1],
    "best_val_balanced_accuracy": float(best[0]),
    "all_results": results,
    "total_min": (time.time() - t0) / 60,
}
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
with open("/kaggle/working/tabpfn_variantD_screen_result.json", "w") as f:
    json.dump(output, f, indent=2)
print("saved /kaggle/working/tabpfn_variantD_screen_result.json")
print(f"total {(time.time()-t0)/60:.1f} min")
